In [177]:
import numpy as np
import pandas as pd
import re

In [178]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [179]:
df = pd.read_csv('C:\\Users\\ACER\Desktop\\Projects\\Machine Learning Based Real Estate System\\Dataset\\Cleaned_properties_v1.csv')

df.head(1)

,property_type,society,sector,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,floorNum,facing,agePossession,nearbyLocations,furnishDetails,features
0,flat,umang winter hills,sector 77,1.0,7500.0,1333.0,Carpet area: 1340 (124.49 sq.m.),2,2,2,not available,3.0,NaN,0 to 1 Year Old,"['Entertainland Mall', 'Delhi Jaipur Expressway', 'Jhankar Senior Secondary School', 'Singhania University, Manesar', 'Miracles Apollo Hospital', 'Indira Gandhi International Airport', 'Garhi Harsaru Junction', 'Eros Corporate Park', 'Hyatt Regency Gurgaon', 'Aravalli Hills']",NaN,"['Security / Fire Alarm', 'Feng Shui / Vaastu Compliant', 'Intercom Facility', 'Lift(s)', 'Maintenance Staff', 'Piped-gas', 'Visitor Parking', 'Swimming Pool', 'Park', 'Security Personnel', 'Fitness Centre / GYM', 'Rain Water Harvesting', 'Club house / Community Center']"


In [180]:
df.shape

(3856, 17)

`Focus is on ->`

`areaWithType, additionalRoom, agePossession, furnishDetails, features`

### 1. areaWithType

In [181]:
df.sample(5)[['price', 'area', 'areaWithType']]

,price,area,areaWithType
3204,1.40,1700.0,Plot area 1700(157.94 sq.m.)
2455,0.49,436.0,Plot area 360(33.45 sq.m.)
967,1.30,1423.0,Carpet area: 1423 (132.2 sq.m.)
699,1.00,1450.0,Carpet area: 1450 (134.71 sq.m.)
778,4.31,1350.0,Plot area 150(125.42 sq.m.)


In [182]:
# This function extracts the Super Built up Area

def get_super_built_up_area(text):
    match = re.search('Super Built up area (\d+\.?\d+)', text)
    if match:
        return float(match.group(1))
    return None

In [183]:
# This function extracts the Built up Area or Carpet Area

def get_area(text, area_type):
    match = re.search(area_type +  r'\s*:\s*(\d+\.?\d*)', text)
    if match:
        return float(match.group(1))
    return None

In [184]:
# This function checks if the area is provided in sq.m and converts it to sqft if needed

def convert_to_sqft(text, area_value):
    if area_value is None:
        return None
    
    match = re.search(r'{} \((\d+\.?\d*) sq.m. \)'.format(area_value), text)
    if match:
        sq_m_value = float(match.group(1))
        return sq_m_value * 10.7639 # Conversion factor from sq.m. to sqft
    return area_value

In [185]:
# Extract Super Built up area and Convert to sqft if Needed
df['super_built_up_area'] = df['areaWithType'].apply(get_super_built_up_area)
df['super_built_up_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['super_built_up_area']), axis=1)

# Extract Build Up area and convert to sqft if needed
df['built_up_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Built Up area'))
df['built_up_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['built_up_area']), axis=1)

# Extract Carpet Area and Convert to sqft if needed
df['carpet_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Carpet area'))
df['carpet_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['carpet_area']), axis=1)

In [186]:
df[['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].sample(5)

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
323,1.16,flat,1743.0,Super Built up area 1743(161.93 sq.m.),1743.0,NaN,NaN
1659,1.07,house,900.0,Plot area 900(83.61 sq.m.),NaN,NaN,NaN
1514,2.00,flat,2300.0,Super Built up area 2300(213.68 sq.m.),2300.0,NaN,NaN
183,3.80,flat,2800.0,Super Built up area 2800(260.13 sq.m.)Built Up area: 2791 sq.ft. (259.29 sq.m.)Carpet area: 2768 sq.ft. (257.16 sq.m.),2800.0,2791.0,2768.0
185,1.55,flat,1578.0,Super Built up area 1578(146.6 sq.m.),1578.0,NaN,NaN


In [187]:
df[~((df['super_built_up_area'].isnull()) | (df['built_up_area'].isnull()) | (df['carpet_area'].isnull()))][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].shape

(536, 7)

In [188]:
df[df['areaWithType'].str.contains('Plot')][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].shape

(710, 7)

In [189]:
df.isnull().sum()

property_type             0
society                   1
sector                    0
price                    18
price_per_sqft           18
area                     18
areaWithType              0
bedRoom                   0
bathroom                  0
balcony                   0
additionalRoom            0
floorNum                 21
facing                 1127
agePossession             1
nearbyLocations         184
furnishDetails         1001
features                661
super_built_up_area    1935
built_up_area          2645
carpet_area            1891
dtype: int64

In [190]:
all_nan_df = df[((df['super_built_up_area'].isnull()) & (df['built_up_area'].isnull()) & (df['carpet_area'].isnull()))][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']]

In [191]:
all_nan_df.head()

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
9,0.75,house,600.0,Plot area 600(55.74 sq.m.),NaN,NaN,NaN
12,5.60,house,3240.0,Plot area 360(301.01 sq.m.),NaN,NaN,NaN
15,0.48,house,80.0,Plot area 80(7.43 sq.m.),NaN,NaN,NaN
19,8.50,house,6300.0,Plot area 6300(585.29 sq.m.),NaN,NaN,NaN
23,0.92,house,603.0,Plot area 67(56.02 sq.m.),NaN,NaN,NaN


In [192]:
all_nan_index = df[((df['super_built_up_area'].isnull()) & (df['built_up_area'].isnull()) & (df['carpet_area'].isnull()))][['price', 'property_type', 'area', 'areaWithType', 'super_built_up_area', 'built_up_area', 'carpet_area']].index

In [193]:
# Function to Extract Plot Area from 'areaWithType' column

def extract_plot_area(area_with_type):
    match = re.search(r'Plot area (\d+\.?\d*)', area_with_type)
    return float(match.group(1)) if match else None

In [194]:
all_nan_df['built_up_area'] = all_nan_df['areaWithType'].apply(extract_plot_area)

In [195]:
all_nan_df.head()

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
9,0.75,house,600.0,Plot area 600(55.74 sq.m.),NaN,600.0,NaN
12,5.60,house,3240.0,Plot area 360(301.01 sq.m.),NaN,360.0,NaN
15,0.48,house,80.0,Plot area 80(7.43 sq.m.),NaN,80.0,NaN
19,8.50,house,6300.0,Plot area 6300(585.29 sq.m.),NaN,6300.0,NaN
23,0.92,house,603.0,Plot area 67(56.02 sq.m.),NaN,67.0,NaN


In [196]:
def convert_scale(row):
    if np.isnan(row['area']) or np.isnan(row['built_up_area']):
        return row['built_up_area']
    else:
        if round(row['area']/row['built_up_area']) == 9.0:
            return row['built_up_area'] * 9
        elif round(row['area']/row['built_up_area']) == 11.0:
            return row['built_up_area'] * 10.7
        else:
            return row['built_up_area']

In [197]:
all_nan_df['built_up_area'] = all_nan_df.apply(convert_scale, axis=1)

In [198]:
all_nan_df.head()

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
9,0.75,house,600.0,Plot area 600(55.74 sq.m.),NaN,600.0,NaN
12,5.60,house,3240.0,Plot area 360(301.01 sq.m.),NaN,3240.0,NaN
15,0.48,house,80.0,Plot area 80(7.43 sq.m.),NaN,80.0,NaN
19,8.50,house,6300.0,Plot area 6300(585.29 sq.m.),NaN,6300.0,NaN
23,0.92,house,603.0,Plot area 67(56.02 sq.m.),NaN,603.0,NaN


In [199]:
# Update original Dataframe

df.update(all_nan_df)

In [200]:
df.isnull().sum()

property_type             0
society                   1
sector                    0
price                    18
price_per_sqft           18
area                     18
areaWithType              0
bedRoom                   0
bathroom                  0
balcony                   0
additionalRoom            0
floorNum                 21
facing                 1127
agePossession             1
nearbyLocations         184
furnishDetails         1001
features                661
super_built_up_area    1935
built_up_area          2079
carpet_area            1891
dtype: int64

In [201]:
df.shape

(3856, 20)

### 2. additionalRoom

In [202]:
df['additionalRoom'].value_counts()

additionalRoom
not available                                    1621
servant room                                      706
study room                                        250
others                                            228
pooja room                                        167
store room                                        103
study room,servant room                           100
pooja room,servant room                            84
pooja room,study room,servant room,store room      72
servant room,others                                60
pooja room,study room,servant room                 55
pooja room,study room,servant room,others          54
servant room,pooja room                            38
servant room,store room                            33
study room,others                                  29
pooja room,study room                              23
pooja room,others                                  17
pooja room,store room                              16
pooja room,st

In [203]:
# List of new columns to be created
new_cols = ['study room', 'servant room', 'store room', 'pooja room', 'others']

# populate the new columns based on the 'additionalRoom' column
for col in new_cols:
    df[col] = df['additionalRoom'].str.contains(col).astype(int)

In [204]:
df.sample(5)[['additionalRoom', 'study room', 'servant room', 'store room', 'pooja room', 'others']]

,additionalRoom,study room,servant room,store room,pooja room,others
988,servant room,0,1,0,0,0
1686,not available,0,0,0,0,0
581,not available,0,0,0,0,0
2035,not available,0,0,0,0,0
389,not available,0,0,0,0,0


### 3. agePossession

In [205]:
df['agePossession'].value_counts()

agePossession
1 to 5 Year Old       1684
5 to 10 Year Old       585
0 to 1 Year Old        536
undefined              343
10+ Year Old           328
Under Construction      90
Within 6 months         70
Within 3 months         26
Dec-23                  20
By 2023                 19
By 2024                 17
Dec-24                  15
Mar-24                  14
Dec-25                   7
Oct-24                   7
Aug-23                   7
Jan-24                   7
Nov-23                   5
Jun-24                   5
Jul-24                   4
Sep-23                   4
Aug-24                   4
By 2025                  4
May-24                   3
Nov-24                   3
Oct-23                   3
Jan-25                   3
Feb-24                   3
Aug-25                   2
Jul-25                   2
Jan-26                   2
Jun 2024                 2
Jun-27                   2
Mar-25                   2
Dec 2023                 2
Jul-27                   2
By 2027       

In [206]:
import re
from datetime import datetime

CURRENT_DATE = datetime.now()  # June 2026 at time of writing

def parse_possession_date(value):
    """Try to parse a possession date string into a datetime object.
    Returns None if it's not a parseable date (e.g. 'undefined', age-range strings)."""
    
    value = value.strip()
    
    # Format: "By 2023", "By 2024", "By 2025", "By 2027"
    match = re.match(r'By (\d{4})', value)
    if match:
        year = int(match.group(1))
        return datetime(year, 12, 31)  # assume end of year
    
    # Format: "Dec-23", "Mar-24", "Jun-27" (Mon-YY)
    match = re.match(r'([A-Za-z]{3})-(\d{2})', value)
    if match:
        month_str, yy = match.group(1), match.group(2)
        try:
            month = datetime.strptime(month_str, '%b').month
            year = 2000 + int(yy)
            return datetime(year, month, 1)
        except ValueError:
            return None
    
    # Format: "Jun 2024", "Dec 2023" (Mon YYYY)
    match = re.match(r'([A-Za-z]{3})\s(\d{4})', value)
    if match:
        month_str, yyyy = match.group(1), match.group(2)
        try:
            month = datetime.strptime(month_str, '%b').month
            year = int(yyyy)
            return datetime(year, month, 1)
        except ValueError:
            return None
    
    return None


def categorize_age_possession(value):
    if pd.isna(value):
        return 'Undefined'
    
    value = str(value).strip()
    
    # --- Existing pre-defined age-range categories ---
    if "0 to 1 Year Old" in value or "Within 6 months" in value or "Within 3 months" in value:
        return 'New Property'
    
    if "1 to 5 Year Old" in value:
        return 'Relatively New'
    
    if "5 to 10 Year Old" in value:
        return 'Moderately New'
    
    if "10+ Year Old" in value:
        return 'Old Property'
    
    if value == "undefined" or value == "Undefined":
        return 'Undefined'
    
    # --- Date-based values: "Dec-23", "By 2025", "Jun 2024", "Under Construction" ---
    if value == "Under Construction":
        return 'Under Construction'
    
    possession_date = parse_possession_date(value)
    
    if possession_date is None:
        return 'Undefined'
    
    if possession_date > CURRENT_DATE:
        return 'Under Construction'
    
    # Possession date has passed - calculate actual age now
    age_years = (CURRENT_DATE - possession_date).days / 365.25
    
    if age_years < 1:
        return 'New Property'
    elif age_years < 5:
        return 'Relatively New'
    elif age_years < 10:
        return 'Moderately New'
    else:
        return 'Old Property'


df['agePossession'] = df['agePossession'].apply(categorize_age_possession)

In [207]:
df['agePossession'].value_counts()

agePossession
Relatively New        1839
New Property           657
Moderately New         585
Undefined              344
Old Property           328
Under Construction     103
Name: count, dtype: int64

### 4. furnishDetails

### 5. features